In [ ]:
# Criando o arquivo useMemes.ts
import os

content = '''import { useState, useEffect, useCallback } from "react";
import { supabase, isSupabaseConfigured, type Meme } from "../lib/supabase";
import toast from "react-hot-toast";

export function useMemes() {
  const [memes, setMemes] = useState<Meme[]>([]);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState<string | null>(null);
  const [favorites, setFavorites] = useState<string[]>([]);

  console.log("🔍 useMemes: Hook iniciando");

  // Carregar memes aprovados
  const loadMemes = useCallback(async () => {
    if (!isSupabaseConfigured || !supabase) {
      console.log("🔍 useMemes: Supabase não configurado");
      setError("Supabase não configurado");
      setLoading(false);
      return;
    }

    try {
      setError(null);
      console.log("🔍 useMemes: Carregando memes do Supabase...");

      const { data, error: fetchError } = await supabase
        .from("memes")
        .select(`
          *,
          category:categories(name),
          profile:profiles(username, full_name)
        `)
        .eq("status", "approved")
        .order("created_at", { ascending: false });

      if (fetchError) {
        console.error("🔍 useMemes: Erro ao buscar memes:", fetchError);
        throw fetchError;
      }

      console.log("🔍 useMemes: Memes encontrados:", data?.length || 0);

      if (data) {
        const transformedMemes = data.map((meme) => ({
          ...meme,
          category: meme.category?.name || "Sem categoria",
          uploaded_by_name: meme.profile?.username || meme.profile?.full_name || "Anônimo",
        }));

        setMemes(transformedMemes);
      }
    } catch (err) {
      console.error("🔍 useMemes: Erro ao carregar memes:", err);
      setError(err instanceof Error ? err.message : "Erro desconhecido");
    } finally {
      setLoading(false);
    }
  }, []);

  // Carregar favoritos do localStorage
  const loadFavorites = useCallback(() => {
    const savedFavorites = localStorage.getItem("memesao_favorites");
    if (savedFavorites) {
      try {
        setFavorites(JSON.parse(savedFavorites));
      } catch (err) {
        console.error("Erro ao carregar favoritos:", err);
        setFavorites([]);
      }
    }
  }, []);

  // Salvar favoritos no localStorage
  const saveFavorites = useCallback((newFavorites: string[]) => {
    localStorage.setItem("memesao_favorites", JSON.stringify(newFavorites));
    setFavorites(newFavorites);
  }, []);

  // Toggle favorito
  const toggleFavorite = useCallback(async (memeId: string) => {
    try {
      const newFavorites = favorites.includes(memeId)
        ? favorites.filter((id) => id !== memeId)
        : [...favorites, memeId];

      saveFavorites(newFavorites);
      
      const isFavorited = newFavorites.includes(memeId);
      toast.success(isFavorited ? "Meme adicionado aos favoritos!" : "Meme removido dos favoritos!");
    } catch (err) {
      console.error("Erro ao atualizar favoritos:", err);
      toast.error("Erro ao atualizar favoritos");
    }
  }, [favorites, saveFavorites]);

  // Download meme
  const downloadMeme = useCallback(async (meme: Meme) => {
    try {
      // Incrementar contador de downloads
      if (supabase) {
        await supabase
          .from("memes")
          .update({ download_count: (meme.download_count || 0) + 1 })
          .eq("id", meme.id);
      }

      // Fazer download da imagem
      const response = await fetch(meme.image_url);
      const blob = await response.blob();
      const url = window.URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = `${meme.title || "meme"}.${meme.format || "jpg"}`;
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      window.URL.revokeObjectURL(url);

      toast.success("Download iniciado!");
    } catch (err) {
      console.error("Erro ao fazer download:", err);
      toast.error("Erro ao fazer download");
    }
  }, []);

  // Compartilhar meme
  const shareMeme = useCallback(async (meme: Meme) => {
    try {
      const shareData = {
        title: meme.title || "Meme do MemesAo",
        text: meme.description || "Confira este meme!",
        url: `${window.location.origin}/meme/${meme.id}`,
      };

      if (navigator.share) {
        await navigator.share(shareData);
      } else {
        // Fallback: copiar URL para clipboard
        await navigator.clipboard.writeText(shareData.url);
        toast.success("Link copiado para a área de transferência!");
      }

      // Incrementar contador de compartilhamentos
      if (supabase) {
        await supabase
          .from("memes")
          .update({ share_count: ((meme as any).share_count || 0) + 1 })
          .eq("id", meme.id);
      }
    } catch (err) {
      console.error("Erro ao compartilhar:", err);
      toast.error("Erro ao compartilhar");
    }
  }, []);

  // Compartilhar com URL
  const shareMemeWithUrl = useCallback(async (meme: Meme) => {
    try {
      const url = `${window.location.origin}/meme/${meme.id}`;
      await navigator.clipboard.writeText(url);
      toast.success("Link copiado para a área de transferência!");
    } catch (err) {
      console.error("Erro ao copiar link:", err);
      toast.error("Erro ao copiar link");
    }
  }, []);

  // Buscar memes
  const searchMemes = useCallback(async (query: string, category?: string) => {
    if (!isSupabaseConfigured || !supabase) {
      return [];
    }

    try {
      let queryBuilder = supabase
        .from("memes")
        .select(`
          *,
          category:categories(name),
          profile:profiles(username, full_name)
        `)
        .eq("status", "approved");

      if (query) {
        queryBuilder = queryBuilder.or(`title.ilike.%${query}%,description.ilike.%${query}%,ocr_text.ilike.%${query}%`);
      }

      if (category) {
        queryBuilder = queryBuilder.eq("category_id", category);
      }

      const { data, error } = await queryBuilder.order("created_at", { ascending: false });

      if (error) throw error;

      return data?.map((meme) => ({
        ...meme,
        category: meme.category?.name || "Sem categoria",
        uploaded_by_name: meme.profile?.username || meme.profile?.full_name || "Anônimo",
      })) || [];
    } catch (err) {
      console.error("Erro ao buscar memes:", err);
      return [];
    }
  }, []);

  // Upload meme
  const uploadMeme = useCallback(async (memeData: any) => {
    if (!isSupabaseConfigured || !supabase) {
      throw new Error("Supabase não configurado");
    }

    try {
      const { data, error } = await supabase
        .from("memes")
        .insert([memeData])
        .select()
        .single();

      if (error) throw error;

      return data;
    } catch (err) {
      console.error("Erro ao fazer upload:", err);
      throw err;
    }
  }, []);

  // Refresh memes
  const refresh = useCallback(() => {
    setLoading(true);
    loadMemes();
  }, [loadMemes]);

  useEffect(() => {
    loadMemes();
    loadFavorites();
  }, [loadMemes, loadFavorites]);

  console.log("🔍 useMemes: Estado atual:", {
    memesLength: memes.length,
    loading,
    error,
    favoritesLength: favorites.length,
  });

  return {
    memes,
    loading,
    error,
    favorites,
    toggleFavorite,
    downloadMeme,
    shareMeme,
    shareMemeWithUrl,
    searchMemes,
    uploadMeme,
    refresh,
  };
}'''

# Escrever o arquivo
os.makedirs("src/hooks", exist_ok=True)
with open("src/hooks/useMemes.ts", "w", encoding="utf-8") as f:
    f.write(content)

print("Arquivo useMemes.ts criado com sucesso!")
